# 04 · Toy fields: Gaussian random fields and their gas cells

A toy field is a gas whose statistics we choose: a Gaussian random field with a known spectrum, isotropic or with the
anisotropy and the acoustic feature of a real survey. It enters the optical depth the same way as a simulation does:
as **gas cells**. The gas is a set of Voronoi cells, in which every point of space belongs to the cell of its nearest
generating point, and inside a cell the gas is uniform. For a toy field the cells are those of its own grid: a regular
grid is already a Voronoi tessellation of its nodes, with cubic cells centred on them. `Cells` holds the generating points, their periodic box and the
cell volumes, and, per absorbing species, what the optical depth needs (review notes, Sec. GasProperties):

| field | unit | required |
|---|---|---|
| `number_density` | comoving cm$^{-3}$ | yes |
| `temperature` | K | yes |
| `velocity` | km/s (peculiar), one vector per cell | yes |
| `turbulence` | m/s ($b_{\rm turb}$) | no (zero) |
| `collisional` | GHz ($A_{\rm coll}$) | no (zero) |

This notebook builds the toy fields of SkewerJAX (`skewerjax.simulations`, a source of cells like a snapshot): a
Gaussian random field (`PowerLawGRF`) and its general model with a BAO feature and line-of-sight anisotropies (§1);
then the gas cells it defines on its grid (§2). The statistics of its draws (the recovered $P(k)$ and $\xi(r)$, and
their anisotropic decompositions) are in notebook 08, §4. Notebook 05 builds the same container from a simulation
snapshot, and both end with the same figure of it. Runtime: ~10 s.

**Requires:** nothing: a Gaussian random field and its cells.  
**Independent of:** lines, cosmology, skewers.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
N_K_3D = 64                       # the 3D grid of §1: (2 N_K_3D)^3 cells

In [ ]:
plot_dag(('samplers', 'fields',)); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

`PowerLawGRF` draws a periodic Gaussian random field $\delta$ in 1, 2 or 3 dimensions. Its statistics are set by one
power spectrum, built in four steps: a power law, an acoustic feature, and two effects of peculiar velocities along the
line of sight.

**1. The power law.** The base spectrum is

$$
P_{\rm pl}(k) = A_s \left(\frac{k}{k_0}\right)^{n_s - 1},
$$

with the amplitude $A_s = P_{\rm pl}(k_0)$ at the pivot $k_0$ and the spectral index $n_s$ (the slope is $n_s - 1$). The
grid is given either by $(L, N)$ or by two of $(k_{\min}, k_{\max}, N_k)$. The normalisation is that of the P1D estimator
of the review notes, $\langle \lvert\delta_k\rvert^2\rangle / V = P(k)$, so `variance()` returns the expected $\sigma^2$.
`sample(key)` is jitted, and `jax.vmap` over keys draws a batch.

**2. The baryon acoustic feature.** Sound waves in the early plasma leave a preferred separation, $r_{\rm BAO}$, between
overdensities. It is a Gaussian bump in the correlation function,

$$
\xi_{\rm BAO}(r) = A_{\rm BAO}\big[g(r - r_{\rm BAO}) + g(r + r_{\rm BAO})\big],\qquad g(u) = e^{-u^2/2\sigma_{\rm BAO}^2},
$$

of height $A_{\rm BAO}$ and width $\sigma_{\rm BAO}$ (the second term is the mirror image that makes it even). It enters
the spectrum through its exact Fourier transform in each dimension, with $G(k) = \sigma_{\rm BAO}\sqrt{2\pi}\, e^{-k^2\sigma_{\rm BAO}^2/2}$:

$$
P_{\rm BAO}^{\rm 1D} = 2A_{\rm BAO}\, G \cos(k r_{\rm BAO}),
\qquad
P_{\rm BAO}^{\rm 3D} = 4\pi A_{\rm BAO}\, G\left[\sigma_{\rm BAO}^2 \cos(k r_{\rm BAO}) + \frac{r_{\rm BAO}\sin(k r_{\rm BAO})}{k}\right],
$$

and in 2D the Hankel transform $2\pi\int_0^\infty r\,\xi_{\rm BAO}(r)\, J_0(kr)\, dr$, by quadrature. In $P(k)$ the bump
becomes oscillations of period $2\pi/r_{\rm BAO}$ in $k$, damped above $k \sim 1/\sigma_{\rm BAO}$.

**3. The Kaiser effect.** In redshift space the coherent infall of matter onto overdensities squeezes them along the line
of sight, which raises the power of the modes along it on large scales:

$$
K(\mu) = \left(1 + \beta\mu^2\right)^2,\qquad \mu = \frac{\lvert k_\parallel\rvert}{\lvert k\rvert},
$$

with $\mu$ the cosine of the angle between the wavevector and the line of sight, a grid axis (`los_axis`, by default the
last one; in 1D, $\mu = 1$), and $\beta$ the strength of the distortion (the growth rate over the bias of the tracer).

**4. The Fingers of God.** Random small-scale velocities smear the gas along the line of sight, which damps the power of
the modes along it on small scales, with one velocity dispersion $\sigma_{\rm FoG}$ (a length):

$$
D(k\mu\,\sigma_{\rm FoG}),\qquad D(x) = \frac{1}{1 + x^2}\ \ \text{(Lorentzian, the default)}\ \ \text{or}\ \ e^{-x^2}\ \ \text{(Gaussian)} .
$$

**The full spectrum** is the product

$$
P(k, \mu) = \big[P_{\rm pl}(k) + P_{\rm BAO}(k)\big]\,\big(1 + \beta\mu^2\big)^2\, D(k\mu\,\sigma_{\rm FoG}) .
$$

- Each term is optional (`A_bao`, `r_bao`, `sigma_bao`; `beta`; `sigma_fog`, `fog`), and all are off by default. The
  simple power law $P(k) = A_s (k/k_0)^{n_s - 1}$ is the special case with them switched off, and its draws are
  unchanged bit for bit.
- A Gaussian field needs $P \ge 0$ at every mode. $K$ and $D$ are never negative, so `PowerLawGRF` checks
  $P_{\rm pl} + P_{\rm BAO} \ge 0$ at construction and raises an error naming the $k$ where it fails: a large bump
  makes the oscillation of $P_{\rm BAO}$ deeper than the power law.
- Every expectation below follows the same terms on the box's own modes: the input $P(k, \mu)$ of every mode in the
  ratios, the maps and the wedges, its inverse transform for $\xi_{\rm in}$, and its average over directions for the guide
  curve of $P(\lvert k\rvert)$.

**The configuration.** One model per dimension, with every term on: $\beta = 1$ and $\sigma_{\rm FoG} = 1$ Mpc, along the
last axis. The base spectra are blue ($n_s = 0.5$ in 1D and 2D, $-0.25$ in 3D), with little large-scale power, so that
the bump stands out in $\xi$. It sits at $r_{\rm BAO} = 16$ Mpc in 1D and 8 Mpc in 2D and 3D, inside the quarter of the
box where $\xi$ is shown, with $A_{\rm BAO}$ at about two thirds of the largest value that keeps $P \ge 0$. The 2D box is
anisotropic ($L_y = 2L_x$). The numbers of draws are set in the setup cell.

In [ ]:
from skewerjax.simulations import PowerLawGRF
rsd = dict(beta=1.0, sigma_fog=1.0)                                                 # Kaiser and Fingers of God, along the last axis
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=100.0, N=1024, A_bao=0.15, r_bao=16.0, sigma_bao=1.6, **rsd)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=(50.0, 100.0), N=(128, 256), A_bao=0.025, r_bao=8.0, sigma_bao=0.8, **rsd)   # y = 2 x
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-0.25, k_min=(2 * np.pi / 50,) * 3, N_k=(N_K_3D,) * 3, A_bao=0.04, r_bao=8.0, sigma_bao=0.8,
                 **rsd)                                                             # grid mode (b): N_k modes per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
try:
    PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=100.0, N=1024, A_bao=1.0, r_bao=16.0, sigma_bao=1.6)
except ValueError as e:
    print("a bump too large:", e)
key = jax.random.PRNGKey(1)
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
iz = d3.shape[2] // 2                                                              # the slice z = L/2
fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.3, 0.75, 1]})
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title=r"1D, $n_s = 0.5$")
Lx, Ly = g2.L
v2, v3 = float(jnp.abs(d2).max()), float(jnp.abs(d3[:, :, iz]).max())             # diverging maps centred on delta = 0
im = axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, Lx, 0, Ly), cmap=CMAPS["diverging"], aspect="equal", vmin=-v2, vmax=v2)
axs[1].set(xlabel="x [Mpc]", ylabel="y (line of sight) [Mpc]", title="2D, box 50 × 100 Mpc")
L3 = g3.L[0]
axs[2].imshow(np.asarray(d3[:, :, iz]).T, origin="lower", extent=(0, L3, 0, L3), cmap=CMAPS["diverging"], aspect="equal", vmin=-v3, vmax=v3)
axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title=f"3D, {d3.shape[0]}³ cells, slice z = L/2")
add_panel_letters(axs)
fig.suptitle("One draw of the general model in 1D, 2D and 3D (BAO, Kaiser and Fingers of God on)", fontsize=21)
plt.show()

## 2 · From the field to gas cells

A toy field gives full control over the statistics of the gas, and `grf_cells` turns it into the same container as a
simulation. `grf_cells(delta, grf, n_bar, temperature=..., velocity=...)` sets $n = \bar n\,(1 + \delta)$ in every
cell, with no transform or clipping ($\bar n$ is the mean density, because $\delta$ has no $k = 0$ mode; the amplitude
must keep $1 + \delta > 0$, and a warning is issued otherwise).

The cells are those of the grid, its own Voronoi-like tessellation: the generating points are the nodes
$x_j = jL/N$, and the cell of each is the cube of side $L/N$ centred on it. So the gas is the realisation itself,
piecewise constant, an exact Gaussian random field at grid resolution, and the deposition finds the cell of a position
from the position itself. Here a 50 cMpc box with $64^3$ cells and $\sigma_\delta = 0.2$.

In [ ]:
from skewerjax.simulations import grf_cells
L4, N4 = 50.0, 64
g4 = PowerLawGRF(A_s=0.04, k_0=2.0, n_s=-1.25, L=(L4,) * 3, N=(N4,) * 3)                    # sigma_delta = 0.2
delta4 = np.asarray(g4.sample(jax.random.PRNGKey(0)))
print(f"sigma_delta = {float(jnp.sqrt(g4.variance())):.2f}, min(1 + delta) = {1 + delta4.min():.3f}; cells of side {L4 / N4:.2f} cMpc")

**The cells, as the optical depth sees them.** A container of cells can carry any temperature and velocity, not only
the density. Here the grid cells take two species, H I and Si III (set manually to 5 times less abundant), the temperature–density
relation of the photo-heated IGM, $T = T_0 (1 + \delta)^{\gamma - 1}$ with $T_0 = 1.5\times10^4$ K and $\gamma - 1 = 0.6$,
and the peculiar velocity of linear theory, the infall onto the overdensities of the same realisation, on the grid:

$$
\mathbf{v}_{\mathbf k} = i\, a H f\, \frac{\mathbf k}{k^2}\, \delta_{\mathbf k},\qquad aHf \simeq 79\ \mathrm{km\,s^{-1}\,cMpc^{-1}}
\ \ (z = 3,\ \Omega_m = 0.31,\ f \simeq 1).
$$

The figure is the standard view of a `Cells` object (`plot_gas_cells`) along one skewer, a `SkewerBundle` of a single
8 cMpc line. `select_cells` keeps the cells within reach of it, the same selection as the optical depth (notebook 07),
and the figure draws those the skewer passes through, in 3D: one row per species; the density, the temperature and
the peculiar velocity (colour: its component along the skewer; arrows: the vector, one per cell from its generating
point). Notebook 05 ends with the same figure for the Voronoi cells of a TNG snapshot, with the same colour maps, view
and skewer direction, so the two can be compared side by side.

In [ ]:
from skewerjax.core import Cosmology, SkewerBundle, phys, select_cells
from skewerjax.utils.plotting import plot_gas_cells
aHf = float(100 * 0.7 * np.sqrt(0.31 * 4 ** 3 + 0.69) / 4)                     # a H(z) f at z = 3 [km/s per cMpc], f ~ 1
k_vec, k_mod = g4.k_grid(real=True)
k2 = np.asarray(k_mod) ** 2
delta_k = np.fft.rfftn(delta4)
v_grid = np.stack([np.fft.irfftn(1j * aHf * np.asarray(kv) / np.where(k2 > 0, k2, 1.0) * delta_k, s=delta4.shape) for kv in k_vec], -1)
toy = grf_cells(delta4, g4, {"HI": 1e-12, "SiIII": 2e-13}, temperature=1.5e4 * (1 + delta4) ** 0.6, velocity=v_grid)
# one skewer of 8 cMpc (the direction and the view of notebook 05), and the cells within its reach
cosmo = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=0.31, Omega_k_0=0.0, Omega_lambda_0=0.69)
start, direction = np.array([20.0, 20.3, 30.4]), np.array([1.0, 0.6, 0.4]) / np.linalg.norm([1.0, 0.6, 0.4])
skewer = SkewerBundle(spatial_points=(start + np.linspace(0.0, 8.0, 64)[:, None] * direction)[None], a0=0.25, x0=0.0,
                      ICs_defined_at_index=0, cosmologies=cosmo)
near = select_cells(toy, skewer.spatial_points)                                # the selection of the optical depth
fig, axs, info = plot_gas_cells(toy, near, skewer)
add_panel_letters(axs)
fig.suptitle("Toy gas cells along one skewer: the grid cubes it crosses, with their density, temperature and velocity", fontsize=21)
plt.show()
print(f"aHf = {aHf:.1f} km/s per cMpc; {info['n_selected']} cells selected, {info['n_crossed']} crossed by the "
      f"{info['path_length']:.0f} cMpc skewer")

In [ ]:
# What a Cells object holds, and how to read it: the same attributes for every source of gas
c = toy
print(c, f"({c.nbytes() / 1e6:.0f} MB)")
print(f"positions {c.positions.shape} [cMpc], volumes {c.volumes.shape} [cMpc^3], size_cMpc = V^(1/3) from "
      f"{c.size_cMpc.min():.2g} to {c.size_cMpc.max():.2g} cMpc, box {c.box} cMpc, species {c.species_keys}")
for sp in c.species_keys:
    f = c.species[sp]                                # the fields of one species: a dict of per-cell arrays
    n, T, v = f["number_density"], f["temperature"], f["velocity"]   # (M,) comoving cm^-3, (M,) K, (M, 3) km/s
    b_turb, A_coll = c.get(sp, "turbulence"), c.get(sp, "collisional")   # optional fields: zeros when left out
    print(f"  {sp:6s} number_density {n.shape}: median {np.median(n):.2e} cm^-3 | temperature {T.shape}: median "
          f"{np.median(T):.3g} K | velocity {v.shape}: rms {np.sqrt((v ** 2).mean()):.0f} km/s | turbulence, collisional: "
          f"{'given' if 'turbulence' in f or 'collisional' in f else 'zero'}")
sub = c.take(np.arange(10)).select_species(c.species_keys[0])   # a subset of the cells, one species: still a Cells
print("subset:", sub)

**This is the object that notebook 05 produces from a simulation snapshot** (`snapshot_cells`), shown there in the same
figure. The inputs could not be more different, a Gaussian random field here and a hydrodynamical snapshot there, but
the result is the same kind of `Cells` object, ready for the deposition: from here on nothing depends on where the cells
came from. Notebook 07 selects the cells along the skewers, cuts the skewers into the pieces of the cells and sums their
profiles into $\tau$.